# TAREFA: Análise de Dados com Groupby, Merge e Tratamento de Nulos

In [2]:
# 1. INSTALAR BIBLIOTECAS NECESSÁRIAS
!pip install pandas numpy -q

# 2. IMPORTAR BIBLIOTECAS
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 3. CRIAR DATASET DE EXEMPLO (com dados reais para prática)

# Tabela 1: Clientes
clientes_data = {
    'id_cliente': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'nome': ['Ana Silva', 'Bruno Souza', 'Carla Lima', 'Daniel Costa',
             'Elena Rocha', 'Fernando Santos', 'Gabriela Alves',
             'Henrique Dias', 'Isabela Nunes', 'João Pedro'],
    'cidade': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'Brasília',
               'Salvador', 'São Paulo', 'Curitiba', 'Porto Alegre',
               'Recife', 'São Paulo'],
    'estado': ['SP', 'RJ', 'MG', 'DF', 'BA', 'SP', 'PR', 'RS', 'PE', 'SP'],
    'segmento': ['Ouro', 'Prata', 'Bronze', 'Ouro', 'Prata',
                  'Ouro', 'Bronze', 'Prata', 'Bronze', 'Ouro'],
    'data_cadastro': pd.date_range('2023-01-01', periods=10, freq='D')
}

# Tabela 2: Vendas (já criar com valores nulos usando dtype=object ou float)
# Solução: Criar listas normais e depois converter para DataFrame
np.random.seed(42)  # Para resultados reproduzíveis

id_venda = list(range(1, 51))
id_cliente = list(np.random.choice(range(1, 11), 50, replace=True))
produto = list(np.random.choice(['Notebook', 'Mouse', 'Teclado', 'Monitor', 'Webcam'], 50))
quantidade = list(np.random.randint(1, 10, 50))
preco_unitario = list(np.random.choice([1500, 50, 120, 800, 200], 50))
data_venda = list(pd.date_range('2024-01-01', periods=50, freq='D'))
status = list(np.random.choice(['Concluído', 'Pendente', 'Cancelado'], 50, p=[0.7, 0.2, 0.1]))

# Adicionar valores nulos (convertendo para float primeiro para aceitar NaN)
quantidade = [float(x) for x in quantidade]  # Converter para float
preco_unitario = [float(x) for x in preco_unitario]  # Converter para float

# Inserir NaN nos índices desejados
for i in [5, 6, 7]:
    quantidade[i] = np.nan

for i in [15, 16, 17]:
    preco_unitario[i] = np.nan

# Para status, manter como string mas adicionar None
status[25] = None

# Criar DataFrame
vendas_data = {
    'id_venda': id_venda,
    'id_cliente': id_cliente,
    'produto': produto,
    'quantidade': quantidade,
    'preco_unitario': preco_unitario,
    'data_venda': data_venda,
    'status': status
}

# Criar DataFrames
df_clientes = pd.DataFrame(clientes_data)
df_vendas = pd.DataFrame(vendas_data)

# 4. VERIFICAR VALORES NULOS
print("="*50)
print("TABELA 1: CLIENTES")
print("="*50)
print(df_clientes.head(10))
print(f"\nShape: {df_clientes.shape}")

print("\n" + "="*50)
print("TABELA 2: VENDAS")
print("="*50)
print(df_vendas.head(10))
print(f"\nShape: {df_vendas.shape}")
print(f"\nValores nulos por coluna:\n{df_vendas.isnull().sum()}")

# 5. IDENTIFICAR E TRATAR VALORES NULOS
print("\n" + "="*50)
print("TRATAMENTO DE VALORES NULOS")
print("="*50)

# Mostrar linhas com nulos
print("\nLinhas com valores nulos (primeiras 5):")
print(df_vendas[df_vendas.isnull().any(axis=1)].head())

# Estratégia 1: Remover linhas com nulos em colunas específicas
df_vendas_sem_nulos = df_vendas.dropna(subset=['quantidade', 'preco_unitario'])
print(f"\n1. Removendo nulos de 'quantidade' e 'preco_unitario':")
print(f"   Antes: {len(df_vendas)} linhas")
print(f"   Depois: {len(df_vendas_sem_nulos)} linhas")

# Estratégia 2: Preencher com valores específicos (recomendado para manter dados)
df_vendas_clean = df_vendas.copy()

# Preencher quantidade com a mediana
df_vendas_clean['quantidade'] = df_vendas_clean['quantidade'].fillna(df_vendas_clean['quantidade'].median())

# Preencher preco_unitario com a média
df_vendas_clean['preco_unitario'] = df_vendas_clean['preco_unitario'].fillna(df_vendas_clean['preco_unitario'].mean())

# Preencher status com 'Desconhecido'
df_vendas_clean['status'] = df_vendas_clean['status'].fillna('Desconhecido')

print(f"\n2. Preenchendo valores nulos:")
print(f"   Quantidade nulos: {df_vendas_clean['quantidade'].isnull().sum()}")
print(f"   Preço unitário nulos: {df_vendas_clean['preco_unitario'].isnull().sum()}")
print(f"   Status nulos: {df_vendas_clean['status'].isnull().sum()}")

# Usar o dataset tratado para análises
df_vendas = df_vendas_clean

# 6. JUNTAR AS TABELAS COM MERGE
print("\n" + "="*50)
print("JUNÇÃO DAS TABELAS (MERGE)")
print("="*50)

# Inner Join (apenas registros que existem em ambas)
df_completo = pd.merge(df_vendas, df_clientes, on='id_cliente', how='inner')
print(f"Inner Join - {len(df_completo)} registros")
print("\nPrimeiras 5 linhas do merge:")
print(df_completo.head())

# Left Join (todos os registros de vendas)
df_left = pd.merge(df_vendas, df_clientes, on='id_cliente', how='left')
print(f"\nLeft Join - {len(df_left)} registros")

# Verificar integridade referencial
print(f"\nClientes sem correspondência: {df_left['nome'].isnull().sum()}")

# 7. AGRUPAMENTOS COM GROUPBY
print("\n" + "="*50)
print("ANÁLISES COM GROUPBY")
print("="*50)

# Calcular valor total da venda
df_completo['valor_total'] = df_completo['quantidade'] * df_completo['preco_unitario']

# Groupby 1: Vendas por produto
print("\n1. Total de vendas por produto:")
vendas_produto = df_completo.groupby('produto').agg({
    'quantidade': 'sum',
    'valor_total': 'sum',
    'id_venda': 'count'
}).rename(columns={'id_venda': 'numero_vendas'})
print(vendas_produto.round(2))

# Groupby 2: Faturamento por estado
print("\n2. Faturamento por estado:")
faturamento_estado = df_completo.groupby('estado')['valor_total'].agg(['sum', 'mean', 'count']).round(2)
print(faturamento_estado.sort_values('sum', ascending=False))

# Groupby 3: Análise por segmento de cliente
print("\n3. Análise por segmento de cliente:")
analise_segmento = df_completo.groupby('segmento').agg({
    'valor_total': ['sum', 'mean', 'std', 'count'],
    'id_cliente': 'nunique'
}).round(2)
print(analise_segmento)

# Groupby 4: Múltiplos níveis (estado + produto)
print("\n4. Vendas por Estado e Produto (Top 10):")
estado_produto = df_completo.groupby(['estado', 'produto'])['valor_total'].sum().sort_values(ascending=False).head(10)
print(estado_produto)

# Groupby 5: Análise temporal (vendas por mês)
df_completo['mes'] = pd.to_datetime(df_completo['data_venda']).dt.month
df_completo['dia_semana'] = pd.to_datetime(df_completo['data_venda']).dt.day_name()

vendas_mes = df_completo.groupby('mes').agg({
    'valor_total': 'sum',
    'quantidade': 'sum'
})
print("\n5. Vendas por mês:")
print(vendas_mes)

# 8. ANÁLISES AVANÇADAS
print("\n" + "="*50)
print("ANÁLISES AVANÇADAS")
print("="*50)

# Top clientes por valor gasto
top_clientes = df_completo.groupby(['id_cliente', 'nome'])['valor_total'].sum().sort_values(ascending=False).head(5)
print("\nTop 5 Clientes por valor gasto:")
print(top_clientes)

# Ticket médio por cidade
print("\nTicket médio por cidade (R$):")
ticket_medio = df_completo.groupby('cidade')['valor_total'].mean().sort_values(ascending=False).round(2)
print(ticket_medio)

# Produtos mais vendidos por estado
print("\nProduto mais vendido (em quantidade) por estado:")
def get_top_produto(df):
    return df.groupby('produto')['quantidade'].sum().idxmax()

produto_mais_vendido = df_completo.groupby('estado').apply(get_top_produto)
print(produto_mais_vendido)

# 9. ESTATÍSTICAS E SUMMARY
print("\n" + "="*50)
print("RESUMO ESTATÍSTICO")
print("="*50)

print("\nEstatísticas gerais das vendas:")
print(df_completo[['quantidade', 'preco_unitario', 'valor_total']].describe())

print("\nDistribuição de status das vendas:")
print(df_completo['status'].value_counts())
print(f"\nPercentual: {df_completo['status'].value_counts(normalize=True).mul(100).round(1)}%")

print("\nClientes por segmento:")
print(df_clientes['segmento'].value_counts())

# 10. ANÁLISES ADICIONAIS COM GROUPBY
print("\n" + "="*50)
print("ANÁLISES ADICIONAIS")
print("="*50)

# Vendas por dia da semana
print("\nVendas por dia da semana:")
vendas_semana = df_completo.groupby('dia_semana')['valor_total'].agg(['sum', 'mean', 'count'])
print(vendas_semana)

# Produto mais lucrativo por segmento
print("\nProduto mais lucrativo por segmento de cliente:")
produto_lucrativo = df_completo.groupby(['segmento', 'produto'])['valor_total'].sum().groupby('segmento').idxmax()
print(produto_lucrativo)

# 11. EXPORTAR RESULTADOS
print("\n" + "="*50)
print("EXPORTANDO RESULTADOS")
print("="*50)

# Salvar resultados principais
df_completo.to_csv('vendas_completas.csv', index=False)
faturamento_estado.to_csv('faturamento_por_estado.csv')
vendas_produto.to_csv('vendas_por_produto.csv')

print("✅ Arquivos exportados com sucesso!")
print("- vendas_completas.csv")
print("- faturamento_por_estado.csv")
print("- vendas_por_produto.csv")

# 12. EXEMPLOS DE FILTROS PRÁTICOS
print("\n" + "="*50)
print("EXEMPLOS DE FILTROS PRÁTICOS")
print("="*50)

# Vendas apenas de clientes Ouro
vendas_ouro = df_completo[df_completo['segmento'] == 'Ouro']
print(f"\n💰 Vendas de clientes Ouro:")
print(f"   Total de registros: {len(vendas_ouro)}")
print(f"   Faturamento: R$ {vendas_ouro['valor_total'].sum():,.2f}")
print(f"   Ticket médio: R$ {vendas_ouro['valor_total'].mean():,.2f}")

# Vendas com valor total > 1000
vendas_acima_1000 = df_completo[df_completo['valor_total'] > 1000]
print(f"\n💎 Vendas acima de R$ 1.000,00:")
print(f"   Quantidade: {len(vendas_acima_1000)} registros")
print(f"   Representa: {(len(vendas_acima_1000)/len(df_completo)*100):.1f}% do total")
if len(vendas_acima_1000) > 0:
    print("\n   Top 5 vendas:")
    print(vendas_acima_1000[['produto', 'valor_total', 'nome', 'cidade']].head())

# Análise de cancelamentos
cancelamentos = df_completo[df_completo['status'] == 'Cancelado']
print(f"\n❌ Análise de cancelamentos:")
print(f"   Total: {len(cancelamentos)} vendas canceladas")
if len(cancelamentos) > 0:
    print(f"   Valor perdido: R$ {cancelamentos['valor_total'].sum():,.2f}")
    print(f"   Produtos mais cancelados: {cancelamentos['produto'].value_counts().index[0]}")

print("\n" + "="*50)
print("✅ TAREFA CONCLUÍDA COM SUCESSO!")
print("="*50)
print("\n📊 APRENDIZADOS PRÁTICOS:")
print("✓ Groupby: agrupamento simples e múltiplo")
print("✓ Merge: junção de tabelas relacionadas")
print("✓ Tratamento de nulos: remoção e preenchimento")
print("✓ Análises estatísticas e filtros")
print("✓ Exportação de resultados")

TABELA 1: CLIENTES
   id_cliente             nome          cidade estado segmento data_cadastro
0           1        Ana Silva       São Paulo     SP     Ouro    2023-01-01
1           2      Bruno Souza  Rio de Janeiro     RJ    Prata    2023-01-02
2           3       Carla Lima  Belo Horizonte     MG   Bronze    2023-01-03
3           4     Daniel Costa        Brasília     DF     Ouro    2023-01-04
4           5      Elena Rocha        Salvador     BA    Prata    2023-01-05
5           6  Fernando Santos       São Paulo     SP     Ouro    2023-01-06
6           7   Gabriela Alves        Curitiba     PR   Bronze    2023-01-07
7           8    Henrique Dias    Porto Alegre     RS    Prata    2023-01-08
8           9    Isabela Nunes          Recife     PE   Bronze    2023-01-09
9          10       João Pedro       São Paulo     SP     Ouro    2023-01-10

Shape: (10, 6)

TABELA 2: VENDAS
   id_venda  id_cliente   produto  quantidade  preco_unitario data_venda  \
0         1           7 

/tmp/ipykernel_33857/2312762387.py:200: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  produto_mais_vendido = df_completo.groupby('estado').apply(get_top_produto)
